In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

In [4]:
PROJECT_ROOT = Path.cwd().parent

c_panel = pd.read_parquet(PROJECT_ROOT / "data" / "clean_panel.parquet")
print(c_panel.shape)
print(c_panel.dtypes)
print(c_panel.head())

(44200, 8)
track_id                          str
track_name                        str
artist_names                      str
rank                            int64
streams                         int64
weeks_since_release           float64
week                   datetime64[us]
release_date           datetime64[us]
dtype: object
                 track_id                                         track_name  \
0  5aAx2yezTd8zXrkmtKl66Z                                            Starboy   
1  7BKLCZ1jbUBVqRi2FVlTVw                                             Closer   
2  5knuzwU65gJK7IF5yJsuaW            Rockabye (feat. Sean Paul & Anne-Marie)   
3  4pdPtRcBmOSQDlJ3Fk945m                                    Let Me Love You   
4  3NdDpSvN911VPGivFlV5d0  I Don’t Wanna Live Forever (Fifty Shades Darke...   

                          artist_names  rank   streams  weeks_since_release  \
0                The Weeknd, Daft Punk     1  25734078                  6.0   
1             The Chainsmokers, 

Defining Lift and Cannibalization Ratio:

Candidate events = specific (track, week) pairs where the track's streams increased by a significant amount (set threshold) compared to previous week. The spike cannot be due to natural post-release hype. 

Lift = how much extra stream growth a treated track has compared to similar control cohort over the same period of time.The control cohort are other tracks with similar rank, maturity (weeks since release) and same week that did NOT experience an unusal spike in streams (event). This ensures we are comparing tracks at a similar stage of life and similar time of the year to avoid external market/ season noise. 

Cannibalization Ratio = artist total growth / track growth over the same post-event window used for lift. A ratio near one would indicate true growth, where the artist's whole catalog grew roughly as much as the spiking track did. A ratio near zero would indicate cannibalization, which means the demand for the track was pulled from the artist's own other tracks, rather than genuine new attention.     


In [23]:
#Week number mapping for candidate event calculation

print(c_panel['week'].nunique()) #221 weeks

unique_weeks = c_panel['week'].unique()
weeks_sorted= np.sort(unique_weeks) 
print(weeks_sorted.dtype)

week_index = {}
for i in range(len(weeks_sorted)):
    week_index[weeks_sorted[i]] = i

c_panel['week_num'] = c_panel['week'].map(week_index)
c_panel.head()


221
datetime64[us]


,track_id,track_name,artist_names,rank,streams,weeks_since_release,week,release_date,week_num
0,5aAx2yezTd8zXrkmtKl66Z,Starboy,"The Weeknd, Daft Punk",1,25734078,6.0,2017-01-06,2016-11-25,0
1,7BKLCZ1jbUBVqRi2FVlTVw,Closer,"The Chainsmokers, Halsey",2,23519705,23.0,2017-01-06,2016-07-29,0
2,5knuzwU65gJK7IF5yJsuaW,Rockabye (feat. Sean Paul & Anne-Marie),"Clean Bandit, Sean Paul, Anne-Marie",3,21216399,11.0,2017-01-06,2016-10-21,0
3,4pdPtRcBmOSQDlJ3Fk945m,Let Me Love You,"DJ Snake, Justin Bieber",4,19852704,22.0,2017-01-06,2016-08-05,0
4,3NdDpSvN911VPGivFlV5d0,I Don’t Wanna Live Forever (Fifty Shades Darke...,"ZAYN, Taylor Swift",5,18316326,4.0,2017-01-06,2016-12-09,0


In [ ]:
#Filtering candidate events

c_panel.sort_values(by = ['track_id', 'week'], inplace = True)
print(c_panel.head())

c_panel['stream_prev'] = c_panel.groupby('track_id')['streams'].shift(1)
c_panel['spike'] = (c_panel['streams'] - c_panel['stream_prev']) / c_panel['stream_prev']
c_panel['post_spike_runway'] = c_panel.groupby('track_id')['week_num'].transform('max') - c_panel['week_num']

c_panel.head()

spike_threshold = 0.3 #decimal increase in streams
post_init_release_window = 8 #number of weeks since release to start looking for a spike to avoid natural post-release hype
min_runway = 8 #number of weeks the track continued to chart after the spike

candidate_events = c_panel[
    (c_panel['spike'] >= spike_threshold) & 
    (c_panel['weeks_since_release'] >=  post_init_release_window) & 
    (c_panel['post_spike_runway'] >= min_runway)
    ]

candidate_events.head()
candidate_events.shape


                    track_id                            track_name  \
2445  000xQL6tZNLJzIrtIgxqSl  Still Got Time (feat. PARTYNEXTDOOR)   
2649  000xQL6tZNLJzIrtIgxqSl  Still Got Time (feat. PARTYNEXTDOOR)   
2851  000xQL6tZNLJzIrtIgxqSl  Still Got Time (feat. PARTYNEXTDOOR)   
3062  000xQL6tZNLJzIrtIgxqSl  Still Got Time (feat. PARTYNEXTDOOR)   
3256  000xQL6tZNLJzIrtIgxqSl  Still Got Time (feat. PARTYNEXTDOOR)   

             artist_names  rank   streams  weeks_since_release       week  \
2445  ZAYN, PARTYNEXTDOOR    46  10037227                  1.0 2017-03-31   
2649  ZAYN, PARTYNEXTDOOR    50   9573086                  2.0 2017-04-07   
2851  ZAYN, PARTYNEXTDOOR    52   8605305                  3.0 2017-04-14   
3062  ZAYN, PARTYNEXTDOOR    63   7663964                  4.0 2017-04-21   
3256  ZAYN, PARTYNEXTDOOR    57   8368114                  5.0 2017-04-28   

     release_date  week_num  stream_prev     spike  post_spike_runway  
2445   2017-03-23        12          NaN    

(293, 12)

In [51]:
candidate_events.head()

,track_id,track_name,artist_names,rank,streams,weeks_since_release,week,release_date,week_num,stream_prev,spike,post_spike_runway
19644,0247StOpd3AkeBQzANX4Zf,Do They Know It's Christmas? - 1984 Version,Band Aid,45,10529841,779.0,2018-12-07,2004-01-01,98,5909362.0,0.781891,108
20211,0247StOpd3AkeBQzANX4Zf,Do They Know It's Christmas? - 1984 Version,Band Aid,12,20989282,782.0,2018-12-28,2004-01-01,101,13423860.0,0.563580,105
30621,0247StOpd3AkeBQzANX4Zf,Do They Know It's Christmas? - 1984 Version,Band Aid,22,20283495,834.0,2019-12-27,2004-01-01,153,10800967.0,0.877933,53
14809,0JP9xo3adEtGSdUEISiszL,Moonlight,XXXTENTACION,10,21727494,14.0,2018-06-22,2018-03-16,74,4888336.0,3.444763,138
1244,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,Bruno Mars,45,8361576,13.0,2017-02-17,2016-11-17,6,4659442.0,0.794544,75


In [ ]:
#Manual review 1 for candidate events

print(c_panel[c_panel['track_id'] == '0247StOpd3AkeBQzANX4Zf'][['track_id','week_num','spike','post_spike_runway']])
print((c_panel[c_panel['track_id'] == '0247StOpd3AkeBQzANX4Zf']).shape)

                     track_id  week_num     spike  post_spike_runway
19521  0247StOpd3AkeBQzANX4Zf        97       NaN                109
19644  0247StOpd3AkeBQzANX4Zf        98  0.781891                108
19836  0247StOpd3AkeBQzANX4Zf        99  0.104494                107
20023  0247StOpd3AkeBQzANX4Zf       100  0.154230                106
20211  0247StOpd3AkeBQzANX4Zf       101  0.563580                105
30052  0247StOpd3AkeBQzANX4Zf       150 -0.576159                 56
30253  0247StOpd3AkeBQzANX4Zf       151  0.117909                 55
30450  0247StOpd3AkeBQzANX4Zf       152  0.086065                 54
30621  0247StOpd3AkeBQzANX4Zf       153  0.877933                 53
40368  0247StOpd3AkeBQzANX4Zf       201 -0.735168                  5
40468  0247StOpd3AkeBQzANX4Zf       202  0.955809                  4
40641  0247StOpd3AkeBQzANX4Zf       203  0.231468                  3
40846  0247StOpd3AkeBQzANX4Zf       204  0.040450                  2
41016  0247StOpd3AkeBQzANX4Zf     

Observation from manual review: there is a gap problem in the data. There are tracks that chart, drop out of Top 200, and re-enter later. This creates two issues:

1) Spike detection can span over a long gap, rather than a week by week comparison. A missing week means the track fell below rank 200 so its streams are unobserved.

2) Lift measurement requires post-spike (candidate event) comparison at every week from t+1 to t+8 to understand whether a track had true incremental growth in the artist's total audience, or a reallocation of streams from the artist's own existing catalog.

Solution: detect weekly gaps in row data and keep only tracks with staying power and drop fast-fading ones. A limitation of this approach is that lift is likely overstated. 

In [140]:
#Re-sort data by track_id and week_num
c_panel = c_panel.sort_values(['track_id', 'week_num']).reset_index(drop=True)

#Define parameters
spike_threshold = 0.3 #decimal increase in streams
post_init_release_window = 8 #number of weeks since release to start looking for a spike to avoid natural post-release hype
min_runway = 8 #number of weeks the track continued to chart after the spike

#Re-calculate spike
c_panel['stream_prev'] = c_panel.groupby('track_id')['streams'].shift(1)
c_panel['spike'] = (c_panel['streams'] - c_panel['stream_prev']) / c_panel['stream_prev']

#Re-calculate post_spike_runway
c_panel['week_num_prev'] = c_panel.groupby('track_id')['week_num'].shift(1)
c_panel['week_num_ahead'] = c_panel.groupby('track_id')['week_num'].shift(-min_runway)
c_panel ['runway'] = c_panel['week_num_ahead'] - c_panel['week_num']

#Filter candidate events
candidate_events = c_panel[
    (c_panel['spike'] >= spike_threshold) & #sufficient spike
    (c_panel['weeks_since_release'] >=  post_init_release_window) & #track is past initial release window
    (c_panel['week_num'] - c_panel['week_num_prev'] == 1) & #previous week is last week, not a gap
    (c_panel['runway'] == min_runway) #there are consecutive weeks of data
    ]

candidate_events.head()

,track_id,track_name,artist_names,rank,streams,weeks_since_release,week,release_date,week_num,stream_prev,spike,post_spike_runway,week_num_prev,week_num_ahead,runway
1405,0JP9xo3adEtGSdUEISiszL,Moonlight,XXXTENTACION,10,21727494,14.0,2018-06-22,2018-03-16,74,4888336.0,3.444763,138,73.0,82.0,8.0
1598,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,Bruno Mars,45,8361576,13.0,2017-02-17,2016-11-17,6,4659442.0,0.794544,75,5.0,14.0,8.0
1599,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,Bruno Mars,20,13283582,14.0,2017-02-24,2016-11-17,7,8361576.0,0.588646,74,6.0,15.0,8.0
1744,0MIcPSq9n2z5H8TAfEstri,Quiero Repetir,"Ozuna, J Balvin",118,5252736,21.0,2018-01-19,2017-08-25,52,3923980.0,0.338625,13,51.0,60.0,8.0
4009,0ofHAoxe9vBkTCp2UQIavz,Dreams - 2004 Remaster,Fleetwood Mac,49,11126560,2279.0,2020-10-09,1977-02-04,194,6931843.0,0.605137,26,193.0,202.0,8.0


In [141]:
#Testing filter criteria on candidate events - all pass
print(candidate_events.shape)
print(candidate_events['spike'].min())  #above set spike threshold                                 
print((candidate_events['weeks_since_release'] < post_init_release_window).sum())  #all tracks past initial release window
print((candidate_events['week_num'] - candidate_events['week_num_prev']).unique()) #spike is from previous week, not a gap
print((candidate_events['runway']).unique()) #consecutive weeks of data available

(146, 15)
0.30069803466327055
0
[1.]
[8.]


In [142]:
#Manual review 2 for candidate events

print(c_panel[c_panel['track_id'] == '0MIcPSq9n2z5H8TAfEstri'][['track_id','week_num','spike','runway']])
print((c_panel[c_panel['track_id'] == '0MIcPSq9n2z5H8TAfEstri']).shape)

                    track_id  week_num     spike  runway
1742  0MIcPSq9n2z5H8TAfEstri        50       NaN     8.0
1743  0MIcPSq9n2z5H8TAfEstri        51  0.091150     8.0
1744  0MIcPSq9n2z5H8TAfEstri        52  0.338625     8.0
1745  0MIcPSq9n2z5H8TAfEstri        53  0.127415     8.0
1746  0MIcPSq9n2z5H8TAfEstri        54  0.004986     8.0
1747  0MIcPSq9n2z5H8TAfEstri        55  0.051567     8.0
1748  0MIcPSq9n2z5H8TAfEstri        56 -0.012823     8.0
1749  0MIcPSq9n2z5H8TAfEstri        57 -0.034907     8.0
1750  0MIcPSq9n2z5H8TAfEstri        58 -0.026942     NaN
1751  0MIcPSq9n2z5H8TAfEstri        59 -0.023244     NaN
1752  0MIcPSq9n2z5H8TAfEstri        60 -0.065375     NaN
1753  0MIcPSq9n2z5H8TAfEstri        61 -0.070670     NaN
1754  0MIcPSq9n2z5H8TAfEstri        62  0.019502     NaN
1755  0MIcPSq9n2z5H8TAfEstri        63 -0.091199     NaN
1756  0MIcPSq9n2z5H8TAfEstri        64 -0.045636     NaN
1757  0MIcPSq9n2z5H8TAfEstri        65 -0.006199     NaN
(16, 15)


In [143]:
print(candidate_events[candidate_events['track_id']== '0MIcPSq9n2z5H8TAfEstri'][['track_id','week_num','spike','runway']])

                    track_id  week_num     spike  runway
1744  0MIcPSq9n2z5H8TAfEstri        52  0.338625     8.0


In [ ]:
#Number of unique tracks in candidate events
unique_candidates = candidate_events['track_id'].nunique()

#35 duplicate rows in candidate events
#These could be consecutive-week flags from one spike that passed the filter criteria (ex: week 52 and 53 both have spike above threshold and min_runway)
#Alternatively, some tracks could have multiple spikes over their lifetime
print(len(candidate_events) - unique_candidates)

#Keeping single largest spike per track
events = candidate_events.loc[candidate_events.groupby('track_id')['spike'].idxmax()]
print(events.shape) #111 unique tracks, 111 rows


35
(111, 15)


In [ ]:
#Manual review 3 for candidate events
review_cols = ['track_name', 'artist_names', 'week', 'streams', 'stream_prev', 'spike', 'weeks_since_release']
print(events.sort_values('spike', ascending=False)[review_cols].head(15))


                                         track_name      artist_names  \
39307                                       changes      XXXTENTACION   
42336                                Jocelyn Flores      XXXTENTACION   
1405                                      Moonlight      XXXTENTACION   
11104                                  Lucid Dreams        Juice WRLD   
20333                                          SAD!      XXXTENTACION   
36398                                    goosebumps      Travis Scott   
16559                                    SICKO MODE      Travis Scott   
24168                                       Falling     Trevor Daniel   
43553                                       Snowman               Sia   
43395           Bohemian Rhapsody - Remastered 2011             Queen   
31852  Another One Bites The Dust - Remastered 2011             Queen   
20507                           HIGHEST IN THE ROOM      Travis Scott   
39252                                          Dior

In [157]:
print(events[events['artist_names'] == 'Sia'][['week', 'weeks_since_release']])

            week  weeks_since_release
43553 2020-12-25                112.0


Observations from manual review of top 15 spikes:

1) Most of the top 15 spikes have a plausible non-promotional cause:
    - XXXTENTACION died in June 2018
    - Juice WRLD died in Dec 2019
    - Pop Smoke's posthumous album was released in July 2020, likely pulling his entire catalog up that week
    - Travis Scott Fortnite event happened in April 2020
    - Snowman by Sia is a holiday song, spiked in Dec 2020
    - Bohemian Rhapsody, the 2018 Queen biopic matches the timing of Queen spikes
    - Falling and Streets were popular TikTok trend songs 
    - Popstar spike is 8 weeks after release so it is likely release momentum

2) Three artists (XXXTENTACION, Travis Scott, Queen) account for 9 out of the 15 top spike events. A Discovery Mode promotion would flag and promote target songs. Artist-level spikes suggest outside demand and the timing aligns with non-promotional events, rather than Discovery Mode boost.

3) The data can't separate causes - an algorithmic boost or spike due to external events will look the same in stream count.

Limitations: I manually reviewd only the top 15 spikes, which can be explained by non-promotional causes. Events further down were not manually reviewed and may show other patterns.  

Conclusion: Spike in streams alone cannot show if a track was promoted through Discovery Mode. The flaw in this proxy study is that the "treated" tracks were picked because they spiked regardless of cause. The controls were picked because they did not spike. By design, the treated group therefore contains track spikes due to outside events (deaths, films, TikTok trends), while the controls were chosen for having no such jump. The gap between the groups reflects how they were selected, not only the effect of a boost. A randomized holdout would be a better approach to assess Discovery Mode boost because in a randomized test the boost is assigned before any outcome is known, so outside events affect both groups about equally.

Next steps: A better measure of Discovery Mode promotional effect would be through a randomized holdout experiment. Eligible tracks would be randomly split, and a small subset would receive no boost. Both groups would be measured over the same weeks, and the difference between the groups would be the effect of the Discovery Mode boost.
